# Проект 1. Аналитическая записка: кешбэк в подписке ПРАЙМ

## Бизнес-вопрос:
Почему доля кешбэка в выручке подписок упала после подорожания 1 марта, и стоит ли из-за этого что-то менять в кешбэке?

## Метрика и зерно:
- Метрика: доля кешбэка в выручке подписок = кешбэк ÷ выручка по успешным платежам
- Зерно: одна строка = месяц × тариф (Базовый или Расширенный)
- Окно: с 1 января по 29 августа 2026 (30 августа выгружено наполовину, 31-го платежей нет)

## Гипотезы:

1. Падение доли кешбэка в выручке подписок после подорожания 1 марта объясняется ростом цены, а не изменением поведения клиентов

2. У заметной части подписок Расширенного тарифа (не меньше 5 % подписок) кешбэк больше платы за подписку (449 ₽), то есть на них сервис теряет деньги.

## ШАГ 1: Подключение

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

## ШАГ 2: Окно, контрольные суммы и таблица «месяц × тариф»

In [ ]:
#Проверка конца периода: в какой день платежи перестают поступать, а кешбэк идёт дальше

print("последний платёж:", q("select max(paid_at) from prime.payments").iloc[0, 0])

a = q("""
    select date_trunc('day', paid_at)::date as day, count(*) as n_payments, round(sum(amount)) as revenue
    from prime.payments where status = 'success' and paid_at >= '2026-08-15'
    group by 1 order by 1
""")
b = q("""
    select date_trunc('day', accrued_at)::date as day, count(*) as n_cashback, round(sum(bonus_amount)) as bonus
    from prime.cashback where accrued_at >= '2026-08-15'
    group by 1 order by 1
""")
display(a.merge(b, on="day", how="outer").sort_values("day"))

In [ ]:
WINDOW = ("2026-01-01", "2026-08-30")   #окно фактов: 1 января – 29 августа 2026, правый конец не включён

print("ЗЕРНО витрины: одна строка — один месяц × один тариф.")
print("Окно:", WINDOW[0], "—", WINDOW[1], "(правый конец не включён: 30.08 выгружен наполовину, 31.08 платежей нет)")

#Контрольные суммы БЕЗ соединений: с ними будем сверять всё, что соберём дальше.
control = q("""
    select
      (select count(*) from prime.payments
        where status = 'success' and paid_at >= :a and paid_at < :b)        as n_payments,
      (select round(sum(amount) * 100) from prime.payments
        where status = 'success' and paid_at >= :a and paid_at < :b)        as revenue_kopecks,
      (select count(*) from prime.cashback
        where accrued_at >= :a and accrued_at < :b)                         as n_cashback,
      (select round(sum(bonus_amount) * 100) from prime.cashback
        where accrued_at >= :a and accrued_at < :b)                         as bonus_kopecks
""", a=WINDOW[0], b=WINDOW[1])

control = control.astype("int64")
CONTROL = control.iloc[0].to_dict()

print(f"успешных платежей:  {CONTROL['n_payments']:>12,}".replace(",", " "))
print(f"выручка:            {CONTROL['revenue_kopecks'] / 100:>12,.2f} ₽".replace(",", " "))
print(f"начислений кешбэка: {CONTROL['n_cashback']:>12,}".replace(",", " "))
print(f"кешбэк:             {CONTROL['bonus_kopecks'] / 100:>12,.2f} ₽".replace(",", " "))
print(f"доля кешбэка в выручке: {CONTROL['bonus_kopecks'] / CONTROL['revenue_kopecks']:.2%}")

In [ ]:
pay_m = q("""
    select to_char(date_trunc('month', p.paid_at), 'YYYY-MM') as month,
           s.tariff_id,
           count(*) as n_payments,
           round(sum(p.amount) * 100)::bigint as revenue_kopecks
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= :a and p.paid_at < :b
    group by 1, 2
    order by 1, 2
""", a=WINDOW[0], b=WINDOW[1])

cb_m = q("""
    select to_char(date_trunc('month', accrued_at), 'YYYY-MM') as month,
           rate_applied::float as rate_applied,
           count(*) as n_cashback,
           round(sum(bonus_amount) * 100)::bigint as bonus_kopecks
    from prime.cashback
    where accrued_at >= :a and accrued_at < :b
    group by 1, 2
    order by 1, 2
""", a=WINDOW[0], b=WINDOW[1])

print("ставки кешбэка в данных:", sorted(cb_m["rate_applied"].unique()))
print("тарифы в платежах:     ", sorted(pay_m["tariff_id"].unique()))

# Сверка с контрольными суммами: части обязаны давать целое.
assert pay_m["n_payments"].sum() == CONTROL["n_payments"], "число платежей не сошлось"
assert pay_m["revenue_kopecks"].sum() == CONTROL["revenue_kopecks"], "выручка не сошлась"
assert cb_m["n_cashback"].sum() == CONTROL["n_cashback"], "число начислений не сошлось"
assert cb_m["bonus_kopecks"].sum() == CONTROL["bonus_kopecks"], "кешбэк не сошёлся"
print("сверка с контрольными суммами: ок")

pay_m.head(6)

## ШАГ 3: Проверка двойного счёта (шеринг)

In [ ]:
naive = q("""
    select count(*) as rows_after,
           round(sum(p.amount) * 100)::bigint as revenue_kopecks
    from prime.payments p
    join prime.subscription_members m on m.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= :a and p.paid_at < :b
""", a=WINDOW[0], b=WINDOW[1])

print(f"платежей до соединения:     {CONTROL['n_payments']:>12,}".replace(",", " "))
print(f"строк после соединения:     {int(naive['rows_after'].iloc[0]):>12,}".replace(",", " "))
print(f"выручка до соединения:      {CONTROL['revenue_kopecks'] / 100:>14,.2f} ₽".replace(",", " "))
print(f"выручка после соединения:   {int(naive['revenue_kopecks'].iloc[0]) / 100:>14,.2f} ₽".replace(",", " "))
print(f"раздулось в {int(naive['revenue_kopecks'].iloc[0]) / CONTROL['revenue_kopecks']:.2f} раза")

## ШАГ 4: Доля кешбэка и разложение изменения (февраль → март)

In [ ]:
TARIFF = {1: "Базовый", 2: "Расширенный"}
RATE_TO_TARIFF = {0.01: 1, 0.03: 2}

cb_m["tariff_id"] = cb_m["rate_applied"].round(2).map(RATE_TO_TARIFF)
assert cb_m["tariff_id"].notna().all(), "есть ставка кешбэка, которой нет в словаре"

m = pay_m.merge(
    cb_m[["month", "tariff_id", "n_cashback", "bonus_kopecks"]],
    on=["month", "tariff_id"], how="left", validate="1:1",
)
assert len(m) == len(pay_m), "строки раздулись или потерялись"
assert m["bonus_kopecks"].sum() == CONTROL["bonus_kopecks"], "кешбэк не сошёлся с контрольной суммой"

m["tariff"] = m["tariff_id"].map(TARIFF)
m["revenue"] = m["revenue_kopecks"] / 100
m["bonus"] = m["bonus_kopecks"] / 100
m["share_pct"] = (m["bonus_kopecks"] / m["revenue_kopecks"] * 100).round(2)

m[["month", "tariff", "n_payments", "revenue", "bonus", "share_pct"]]

In [ ]:
def part(month):
    d = m[m["month"] == month].sort_values("tariff_id")
    n = d["n_payments"].to_numpy(float)          # число платежей по тарифам
    c = (d["bonus"] / d["n_payments"]).to_numpy()   # кешбэк на один платёж
    f = (d["revenue"] / d["n_payments"]).to_numpy() # средний чек (цена)
    return n, c, f

def share(n, c, f):
    """Доля кешбэка в выручке по обоим тарифам вместе."""
    return (n * c).sum() / (n * f).sum()

n0, c0, f0 = part("2026-02")
n1, c1, f1 = part("2026-03")

s0 = share(n0, c0, f0)                  # февраль, как есть
s1 = share(n1, c0, f0)                  # + структура платежей мартовская
s2 = share(n1, c1, f0)                  # + кешбэк на платёж мартовский
s3 = share(n1, c1, f1)                  # + цена мартовская = март, как есть

mar = m[m["month"] == "2026-03"]
assert abs(s3 - mar["bonus"].sum() / mar["revenue"].sum()) < 1e-12, "цепочка не пришла к марту"

effects = pd.Series({
    "структура платежей по тарифам": (s1 - s0) * 100,
    "кешбэк на платёж":              (s2 - s1) * 100,
    "цена (средний чек)":            (s3 - s2) * 100,
})
print(f"доля в феврале: {s0 * 100:.2f}%   в марте: {s3 * 100:.2f}%   изменение: {(s3 - s0) * 100:+.2f} п.п.")
print(effects.round(2).to_string())
assert abs(effects.sum() - (s3 - s0) * 100) < 1e-9, "части не сходятся с целым"

# Порядок подстановок — часть ответа. Проверим другой порядок: цена первой.
t1 = share(n0, c0, f1); t2 = share(n0, c1, f1); t3 = share(n1, c1, f1)
print("\nдругой порядок (цена -> кешбэк -> структура):")
print(pd.Series({"цена": (t1 - s0) * 100, "кешбэк на платёж": (t2 - t1) * 100,
                 "структура": (t3 - t2) * 100}).round(2).to_string())

## ШАГ 5: Графики 1–3

In [ ]:
import matplotlib.pyplot as plt

RED, GREEN, GREY, LIGHT = "#C1272D", "#2A9D8F", "#6B7280", "#9CA3AF"
LAST_MONTH = "2026-08"      #если проверка покажет, что август неполный, поставьте "2026-07"


def ru(x, fmt="{:.2f}"):
    """Число с запятой и настоящим минусом: как на слайдах Л6."""
    return fmt.format(x).replace(".", ",").replace("-", "−")


# График 1. Как менялась доля кешбэка: линия, ось от нуля, подписи у линий вместо легенды.
lines = m.pivot_table(index="month", columns="tariff", values="share_pct")
lines = lines.loc[lines.index <= LAST_MONTH]
march = list(lines.index).index("2026-03")

fig, ax = plt.subplots(figsize=(9, 4.6), layout="constrained")
for tariff, color in [("Базовый", LIGHT), ("Расширенный", RED)]:
    ax.plot(lines.index, lines[tariff], marker="o", color=color)
    ax.annotate(tariff, (len(lines) - 1, lines[tariff].iloc[-1]), textcoords="offset points",
                xytext=(8, 0), va="center", color=GREY if tariff == "Базовый" else RED)
ax.axvline(march - 0.5, color="#111827", linestyle="--", linewidth=1)
ax.annotate("1 марта: подорожание", (march - 0.5, 5.7), textcoords="offset points",
            xytext=(6, 0), va="center", color="#111827")
ax.margins(x=0.12)
ax.set(ylim=(0, 6), ylabel="кешбэк, % от выручки")
ax.set_xlabel("месяц 2026 года")
ax.set_title(f"После подорожания доля кешбэка в выручке упала с {ru(s0 * 100, '{:.1f}')} % до {ru(s3 * 100, '{:.1f}')} %",
             loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
plt.show()


# График 2. Из чего сложилось изменение доли: каскад (февраль -> март), в п.п.
def waterfall(deltas: pd.Series, title: str) -> plt.Axes:
    """Каскад изменений: по столбику на фактор и «итого» в конце."""
    before = deltas.cumsum() - deltas      # уровень, на котором закончился предыдущий столбик
    colors = [RED if v < 0 else GREEN for v in deltas]

    fig, ax = plt.subplots(figsize=(9, 4.6), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color=GREY, zorder=2)
    label = lambda v: "≈ 0" if abs(v) < 0.005 else ru(v, "{:+.2f}")
    for group in (bars, total):
        ax.bar_label(group, fmt=label, padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("п.п. доли кешбэка в выручке")
    ax.use_sticky_edges = False
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
    return ax


short = effects.rename({"структура платежей по тарифам": "структура\nтарифов",
                        "кешбэк на платёж": "кешбэк\nна платёж",
                        "цена (средний чек)": "цена"})
change = (s3 - s0) * 100
ax = waterfall(short, title=f"Доля кешбэка упала на {ru(abs(change))} п.п. — почти целиком из-за цены")
ax.set_xlabel("Февраль → март 2026. Порядок подстановок: структура, кешбэк, цена. "
              "Сумма подписей может отличаться на 0,01 из-за округления.")
plt.show()


# График 3. Что выросло сильнее: столбики от нуля, числа над столбиками.
growth = pd.DataFrame({"цена платежа": (f1 / f0 - 1) * 100,
                       "кешбэк на платёж": (c1 / c0 - 1) * 100},
                      index=["Базовый", "Расширенный"])

fig, ax = plt.subplots(figsize=(8, 4.4), layout="constrained")
growth.plot.bar(ax=ax, color=[RED, LIGHT], rot=0)
for container in ax.containers:
    ax.bar_label(container, fmt=lambda v: ru(v, "{:.1f}") + " %", padding=2)
ax.axhline(0, color="#111827", linewidth=0.8)
ax.set_title("Платёж подорожал сильнее, чем вырос кешбэк на платёж", loc="left")
ax.set(ylabel="рост, % (март к февралю)")
ax.set_xlabel("тариф")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
plt.show()

## ШАГ 6: Гипотеза 1 - убыточные подписки и график 4

In [ ]:
subs_all = q("select subscription_id, owner_client_id from prime.subscriptions where tariff_id = 2")
mem_all = q("""
    select m.subscription_id, m.client_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.tariff_id = 2
""")

# Шеринг у Расширенного тарифа: сколько участников у подписки?
k = mem_all.groupby("subscription_id").size()                           # участников у каждой подписки
n_members = subs_all["subscription_id"].map(k).fillna(0).astype(int)    # у подписок без участников будет 0
print(n_members.value_counts().sort_index())

# Шеринг у Расширенного тарифа: входит ли владелец подписки в список участников?
pairs = subs_all.merge(mem_all, left_on=["subscription_id", "owner_client_id"],
                       right_on=["subscription_id", "client_id"], how="inner")
print("подписок:", len(subs_all), " владелец есть среди участников:", pairs["subscription_id"].nunique())

In [ ]:
FEE = 449   # плата за подписку Расширенного после подорожания, ₽

# Гипотеза 1 для одного месяца: сколько кешбэка набегает на каждую подписку Расширенного.
def check_month(a, b):
    # подписки Расширенного с успешным платежом в месяце и их владельцы
    subs2 = q("""
        select distinct s.subscription_id, s.owner_client_id as client_id
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        where s.tariff_id = 2 and p.status = 'success'
          and p.paid_at >= :a and p.paid_at < :b
    """, a=a, b=b)
    # участники этих подписок, которые состояли в них в течение месяца
    members = q("""
        select m.subscription_id, m.client_id
        from prime.subscription_members m
        join prime.subscriptions s on s.subscription_id = m.subscription_id
        where s.tariff_id = 2 and m.joined_at < :b
          and (m.left_at is null or m.left_at >= :a)
    """, a=a, b=b)
    # кешбэк каждого клиента за месяц
    bonus = q("""
        select t.client_id, sum(c.bonus_amount)::float as bonus
        from prime.cashback c
        join prime.transactions t on t.transaction_id = c.transaction_id
        where c.accrued_at >= :a and c.accrued_at < :b
        group by t.client_id
    """, a=a, b=b)
    members = members[members["subscription_id"].isin(subs2["subscription_id"])]
    # все люди подписки: владелец и участники; повторы убираем (как union в SQL)
    ppl = pd.concat([subs2, members], ignore_index=True).drop_duplicates()
    # validate="m:1": у клиента в bonus ровно одна строка
    people = ppl.merge(bonus, on="client_id", how="left", validate="m:1")
    people["bonus"] = people["bonus"].fillna(0)      # у кого кешбэка не было, там ноль
    assert len(people) == len(ppl), "строки раздулись или потерялись"
    # сворачиваем до подписки: сколько людей и сколько кешбэка у всех вместе
    per_sub = people.groupby("subscription_id").agg(n_people=("client_id", "size"), bonus=("bonus", "sum"))
    return people, per_sub

In [ ]:
%%time
# Проверка по месяцам март–июль, завивисит ли вывод от выбранного месяца
months = ["2026-03", "2026-04", "2026-05", "2026-06", "2026-07"]
edges = ["2026-03-01", "2026-04-01", "2026-05-01", "2026-06-01", "2026-07-01", "2026-08-01"]
rows = []
for i, month in enumerate(months):
    people, per_sub = check_month(edges[i], edges[i + 1])
    n_loss = int((per_sub["bonus"] > FEE).sum())
    rows.append((month, len(per_sub), n_loss))
res = pd.DataFrame(rows, columns=["month", "n_subs", "n_loss"])
res["share_pct"] = res["n_loss"] / res["n_subs"] * 100
print(res)
# после цикла people и per_sub относятся к июлю: это последний месяц в списке

In [ ]:
# Итоги за июль — чтобы описать типичную подписку и оценить масштаб превышения.
MONTH = ("2026-07-01", "2026-08-01")
multi = people.groupby("client_id").size()    # в скольких подписках человек; если больше одной, его кешбэк в каждой (оценка сверху)
print("подписок Расширенного с платежом в июле:", len(per_sub))
print("людей на подписке в среднем:", round(per_sub["n_people"].mean(), 2))
print("клиентов в нескольких подписках сразу:", int((multi > 1).sum()), "из", len(multi))
print("кешбэк на подписку: медиана", round(per_sub["bonus"].median(), 1),
      " p90", round(per_sub["bonus"].quantile(0.9), 1),
      " максимум", round(per_sub["bonus"].max(), 1))
n_loss = int((per_sub["bonus"] > FEE).sum())
print(f"подписок, где кешбэк больше платы {FEE} ₽: {n_loss} ({n_loss / len(per_sub):.2%})")

In [ ]:
# Сколько рублей сверх платы кешбэк «съедает» у этих подписок в июле — чтобы оценить масштаб потерь.
excess = (per_sub["bonus"] - FEE).clip(lower=0)
july_rev = m.loc[(m["month"] == "2026-07") & (m["tariff"] == "Расширенный"), "revenue"].iloc[0]
print(f"кешбэк сверх платы {FEE} ₽ в июле: {excess.sum():,.0f} ₽ у {n_loss} подписок".replace(",", " "))
print(f"выручка Расширенного в июле: {july_rev:,.0f} ₽; сверхкешбэк = {excess.sum() / july_rev:.3%} от неё".replace(",", " "))

In [ ]:
# График 4. Как распределён кешбэк на подписку — чтобы показать, что превышение платы встречается редко.
bins = [0, 50, 150, 300, FEE, float("inf")]
names = ["до 50", "50–150", "150–300", f"300–{FEE}", f"больше {FEE}"]
dist = pd.cut(per_sub["bonus"], bins=bins, labels=names, include_lowest=True).value_counts().reindex(names)

fig, ax = plt.subplots(figsize=(8, 4.4), layout="constrained")
bars = ax.bar(dist.index, dist.values, color=[LIGHT] * 4 + [RED], zorder=2)
ax.bar_label(bars, labels=[ru(v, "{:,.0f}").replace(",", " ") for v in dist.values], padding=3)
ax.set_title(f"Кешбэк больше платы за подписку лишь у {n_loss} из {len(per_sub):,} подписок Расширенного".replace(",", " "),
             loc="left")
ax.set(ylabel="число подписок")
ax.set_xlabel("кешбэк на подписку за июль 2026, ₽ (владелец и участники вместе)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
plt.show()

## ШАГ 7: Вывод и рекомендация
**Ответ на вопрос:**

**Почему?** Доля кешбэка в выручке подписок упала с 4,33% до 3,72% после подорожания 1 марта почти полностью из-за цены. Так как платеж подорожал (на 25 % у Базового и на 12,5 % у Расширенного), а кешбэк почти не изменился на один платеж (рост на 1-2%), то есть клиенты тратили примерно столько же, как и раньше, но доля кешбэка в выручке стала меньше.

**Нужно ли менять что-то с кешбэком?** Нет, так как после проверки видно, что подписок, где кешбэк превышает плату только 29 из 71 794 (0,04 %) и этот кешбэк сверх платы выходит только 2279 руб. в июле в сравнении с 33 млн. выручки расширенного тарифа это незаметно, следовательно урезать кешбэк ради этого не стоит, так как сервис почти не теряет деньги. 

**Гипотеза 1:** Подтвердилась. Падение доли кешбэка в выручке подписок с 4,33 % до 3,72 %, то есть на 0,61 п.п. после подорожания 1 марта объясняется ростом цены, а не изменением поведения. По данным вклад цены, который составил −0,68, в сравнении с кешбэком на платёж и структурой (+0,06 п.п. и около 0), самый большой, следовательно его влияние тоже больше остальных вместе.

**Гипотеза 2:** Не подтвердилась. По данным только у 0,04 % (29 из 71794) подписок Расширенного тарифа, по которым был платёж в июле, кешбэк превысил плату за подписку (449 ₽). В сравнении, все другие месяцы дали похожий процент ниже 5 % (март–июль: 0,02–0,05 %). То есть кешбэк сверх платы равнялся 2 279 руб. в июле, что составляет всего 0,007 % выручки Расширенного). 

**Ограничения** 

1. Так как один клиент может быть в нескольких подписках, его кешбэк мог быть посчитан в каждой, поэтому 29 это максимум, но по сути таких подписок может быть меньше
2. Гипотеза 1 проверена только для Расширенного тарифа, у Базового тоже есть шеринг (до 3 участников), его не проверяли
3. Кешбэк мы отнесли к тарифу по ставке (1 % → Базовый, 3 % → Расширенный)
4. Причина падения платежей примерно на 15 % с 22.08 при прежнем кешбэке в день не проверена. Кешбэк тот же, а доля в августе чуть выше, чем в июле.
5. Август взят не полностью, окно данных заканчивается 29.08, так как 30.08 загружен наполовину, 31.08 без платежей

### Проверка веса данных в памяти: сколько занимают основные таблицы ноутбука.

In [ ]:
#memory_usage(deep=True) считает и сами строки (без deep=True pandas занижает вес).
def mb(df):
    return df.memory_usage(deep=True).sum() / 1e6

for name, df in [("pay_m", pay_m), ("cb_m", cb_m), ("m", m), ("people", people), ("per_sub", per_sub)]:
    print(f"{name:8} {len(df):>8} строк  {mb(df):>8.2f} МБ")

### Дополнительно: шеринг у Базового тарифа

In [ ]:
# Cмотрим, сколько участников у подписки Базового, чтобы понять, есть ли у него шеринг на нескольких человек.
print(q("""
    select n_members, count(*) as subscriptions
    from (
        select s.subscription_id, count(m.client_id) as n_members
        from prime.subscriptions s
        left join prime.subscription_members m on m.subscription_id = s.subscription_id
        where s.tariff_id = 1
        group by s.subscription_id
    ) t
    group by n_members order by n_members
"""))

In [ ]:
## Проверка допущения: кешбэк = сумма транзакции × ставка. Для того чтобы убедиться, что он зависит от суммы покупок.
display(q("""
    select t.status,
           count(*) as n,
           sum((abs(c.bonus_amount - t.amount * c.rate_applied) < 0.015)::int) as n_match
    from prime.cashback c
    join prime.transactions t on t.transaction_id = c.transaction_id
    group by t.status order by t.status
"""))